In [ ]:
from huggingface_hub import login

# Login with your Hugging Face API token
login(token="your_huggingface_token")


In [2]:
!pip install -q transformers datasets

In [3]:
import gc
import torch

torch.cuda.empty_cache()
gc.collect()


0

In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForLanguageModeling
from datasets import load_dataset
from accelerate import Accelerator
import torch
import gc

# ✅ Clear memory
gc.collect()
torch.cuda.empty_cache()

# ✅ Device setup
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using", device)

# ✅ Load tokenizer and model
model_name = "bigscience/bloom-560m"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# ✅ Set pad token (important!)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32,  # Avoid fp16 instability
    use_cache=False
)

model.gradient_checkpointing_enable()
model.train()
model.to(device)

# ✅ Load dataset
dataset = load_dataset("text", data_files={"train": "/kaggle/input/mini-tel/mini_tel.txt"})

# ✅ Tokenize
def tokenize_function(example):
    tokens = tokenizer(example["text"], truncation=True, max_length=128)
    if len(tokens["input_ids"]) == 0:
        return None
    return tokens

tokenized = dataset.map(tokenize_function, batched=True, remove_columns=["text"])
tokenized = tokenized.filter(lambda x: len(x["input_ids"]) > 0)

# ✅ Add labels and attention masks
def preprocess(example):
    example["labels"] = example["input_ids"]
    example["attention_mask"] = [1] * len(example["input_ids"])
    return example

tokenized = tokenized.map(preprocess)

# ✅ Set correct format
tokenized.set_format(
    type="torch",
    columns=["input_ids", "labels", "attention_mask"],
    dtype=torch.long
)

train_dataset = tokenized["train"]

# ✅ Data collator
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# ✅ Dataloader
from torch.utils.data import DataLoader
train_dataloader = DataLoader(
    train_dataset,
    shuffle=True,
    collate_fn=data_collator,
    batch_size=1
)

# ✅ Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

# ✅ Accelerator (no mixed precision)
accelerator = Accelerator(mixed_precision="no")
model, optimizer, train_dataloader = accelerator.prepare(model, optimizer, train_dataloader)

# ✅ Training loop
epochs = 3
model.train()

for epoch in range(epochs):
    print(f"\nEpoch {epoch + 1}")
    total_loss = 0
    for step, batch in enumerate(train_dataloader):
        batch = {k: v.long() for k, v in batch.items()}

        outputs = model(**batch)
        loss = outputs.loss

        # ✅ Check for NaN
        if torch.isnan(loss):
            print(f"⚠️ NaN loss at step {step}")
            continue

        accelerator.backward(loss)

        # ✅ Clip gradients to prevent instability
        accelerator.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        optimizer.zero_grad()

        total_loss += loss.item()

        if step % 50 == 0:
            print(f"Step {step}, Loss: {loss.item():.4f}")

    avg_loss = total_loss / len(train_dataloader)
    print(f"Epoch {epoch + 1} Loss: {avg_loss:.4f}")

# ✅ Save the fine-tuned model
model.save_pretrained("/kaggle/working/bloom560m-finetuned")
tokenizer.save_pretrained("/kaggle/working/bloom560m-finetuned")


Using cuda


tokenizer_config.json:   0%|          | 0.00/222 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

...

Map:   0%|          | 0/3393 [00:00<?, ? examples/s]

Filter:   0%|          | 0/3393 [00:00<?, ? examples/s]

Map:   0%|          | 0/3098 [00:00<?, ? examples/s]


Epoch 1
Step 0, Loss: 8.7784
Step 50, Loss: 3.6782
Step 100, Loss: 5.4098
Step 150, Loss: 4.3969
⚠️ NaN loss at step 154
Step 200, Loss: 5.9134
Step 250, Loss: 3.4438
Step 300, Loss: 4.6205
Step 350, Loss: 2.5575
Step 400, Loss: 3.5368
Step 450, Loss: 4.5873
Step 500, Loss: 5.1993
Step 550, Loss: 4.9913
Step 600, Loss: 4.1803
Step 650, Loss: 4.5915
Step 700, Loss: 4.8279
Step 750, Loss: 3.4805
Step 800, Loss: 4.3305
Step 850, Loss: 6.1819
Step 900, Loss: 2.8303
Step 950, Loss: 2.9603
Step 1000, Loss: 3.7055
Step 1050, Loss: 4.2746
Step 1100, Loss: 4.1377
Step 1150, Loss: 3.9543
Step 1200, Loss: 4.0865
Step 1250, Loss: 4.8263
Step 1300, Loss: 4.9943
Step 1350, Loss: 2.9087
Step 1400, Loss: 3.5553
Step 1450, Loss: 2.1535
Step 1500, Loss: 3.8203
Step 1550, Loss: 4.0978
Step 1600, Loss: 4.0985
Step 1650, Loss: 1.3022
Step 1700, Loss: 4.2558
Step 1750, Loss: 3.4757
Step 1800, Loss: 3.2487
Step 1850, Loss: 3.5633
Step 1900, Loss: 3.5902
Step 1950, Loss: 4.7265
Step 2000, Loss: 1.7178
Step 2

('/kaggle/working/bloom560m-finetuned/tokenizer_config.json',
 '/kaggle/working/bloom560m-finetuned/special_tokens_map.json',
 '/kaggle/working/bloom560m-finetuned/tokenizer.json')

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Load the fine-tuned model and tokenizer
model_path = "/kaggle/working/bloom560m-finetuned"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32)

# Input Telugu prompt (seed text)
input_text = "సాధారణంగా నైరుతి బుతుపవనాలు జూన్‌ మొదటి వారంలో"

# Tokenize the input
inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

# Generate continuation
outputs = model.generate(
    **inputs,
    max_new_tokens=50,              # Generate up to 100 new tokens
    do_sample=True,                  # Enable sampling (for more diverse output)
    temperature=0.7,                 # Sampling temperature
    top_k=50,                        # Top-k sampling
    top_p=0.95,                      # Nucleus sampling
    repetition_penalty=1.2,          # Penalize repeating same phrases
    pad_token_id=tokenizer.eos_token_id,  # Avoid padding errors
    eos_token_id=tokenizer.eos_token_id   # Stop early if EOS token is generated
)

# Decode and print the generated text
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("\nGenerated Text:\n")
print(generated_text)


Generated Text:

సాధారణంగా నైరుతి బుతుపవనాలు జూన్‌ మొదటి వారంలో వుంచుకొని జూన్‌ రెండవ పక్షం నుంచి సెప్టెంబర్‌ మొదటి వారం లోపల ప్రవేశిస్తాయి. జులై-ఆగస్టులో నైరుతి బుతుపవనాల ప్రభావం వలన సెప్టెంబరు రెండవ పక్షములో కూడా ఆశించి కొంతమ నష్టాన్ని కలిగించ


In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import math

model_path = "/kaggle/working/bloom560m-finetuned"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path)
model.eval()

text = "ఆంధ్రప్రదేశ్‌ ప్రధానంగా వ్యవసాయ ఆధారిత రాష్ట్రం. రాష్ట్రంలోని పంటల విస్తీర్ణం, పంటల సరళి "
inputs = tokenizer(text, return_tensors="pt")
with torch.no_grad():
    outputs = model(**inputs, labels=inputs["input_ids"])
    loss = outputs.loss
    perplexity = math.exp(loss.item())

print(f"Perplexity: {perplexity:.2f}")


Perplexity: 6.59


In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Load your fine-tuned model and tokenizer
model_path = "/kaggle/input/finetuned/bloom560m-finetuned/kaggle/working/bloom560m-finetuned"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32)
model.to("cuda" if torch.cuda.is_available() else "cpu")
model.eval()

# 💬 Your question (you can change this!)
your_question = "2022లో ఆంధ్రప్రదేశ్‌లో సాధారణ నైరుతి బుతుపవనాల వర్షపాతం ఎంత?"

# Encode the question
input_ids = tokenizer.encode(your_question, return_tensors="pt").to(model.device)

# Generate the response
output_ids = model.generate(
    input_ids,
    max_length=150,
    num_return_sequences=1,
    do_sample=True,
    temperature=0.7,
    top_p=0.95,
    eos_token_id=tokenizer.eos_token_id,  # 🛑 This tells it when to stop
    pad_token_id=tokenizer.pad_token_id
)

# Decode and print the output
generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print("\n📝 Generated Answer:\n", generated_text)


📝 Generated Answer:
 2022లో ఆంధ్రప్రదేశ్‌లో సాధారణ నైరుతి బుతుపవనాల వర్షపాతం ఎంత? ఆంధ్రప్రదేశ్‌ రాష్ట్రంలో రైతులు వర్షాధారంగా పండించే పండ్ల తోటల్లో సాధారణంగా జూన్‌ - జూలై నెలల్లో వర్షపాత అధికంగా నమోదు అయినప్పటికి జూన్‌ - జూలై నెలల్లో నైరుతి బుతుపవనాల ప్రవేశం ఎక్కువగా నమోదైనది. ఆంధ్రప్రదేశ్‌ రాష్ట్రంలో జూన్‌ - జూలై నెలల్లో నైరుతి బుతుపవనాల వర్షపాతాన్ని గమనించినప్పుడు ఆంధ్రప్రదేశ్‌ రాష్ట్రంలో కురుస్తా సాధారణ వర్షపాత నమోదయ్యే అవకాశం ఉందని దీర్హకాలిక ముందస్తు ముందస్తు ముందస్తు సూచనలు ఇవ్వడం జరిగినది. ఆంధ్రప్రదేశ్‌ రాష్ట్రంలో జూన్‌ - జూలై నెలల్లో నైరుతి బుతుపవనాల వర్షపాతాన్ని గమనించినప్పుడు ఆంధ్రప్రదేశ్‌


In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_path = "/kaggle/input/finetuned/bloom560m-finetuned/kaggle/working/bloom560m-finetuned"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")


In [30]:
import faiss
import numpy as np

# ✅ Convert embeddings list to NumPy array
embedding_array = np.array(embeddings).astype("float32")

# ✅ Create FAISS index (L2 or cosine-based, we'll use L2 here)
dimension = embedding_array.shape[1]  # should be 768
index = faiss.IndexFlatL2(dimension)

# ✅ Add embeddings to the index
index.add(embedding_array)
print(f"🔎 FAISS index built with {index.ntotal} vectors.")

🔎 FAISS index built with 2333 vectors.


In [4]:
faiss.write_index(index, "telugu_index.faiss")

NameError: name 'faiss' is not defined

In [3]:
!pip install -q faiss-cpu sentence-transformers


In [6]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# ✅ 1. Load the Telugu text file
file_path = "/kaggle/input/mini-tel/mini_tel.txt"  # Update with actual path

with open(file_path, "r", encoding="utf-8") as f:
    telugu_text = f.read()

print("🔹 Sample of the loaded text:")
print(telugu_text[:500])


# ✅ 2. Split into chunks
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,      # you can adjust chunk size as needed
    chunk_overlap=50,    # small overlap for better context continuity
)

chunks = text_splitter.split_text(telugu_text)

print(f"🔹 Total chunks created: {len(chunks)}")


# ✅ Load the multilingual E5 model
model = SentenceTransformer("intfloat/multilingual-e5-base")


# Replace with your actual folder name
index = faiss.read_index("/kaggle/input/fiass-index/telugu_index.faiss")

print(f"✅ Index loaded with {index.ntotal} vectors")

# Example Telugu query

query = "2022లో ఆంధ్రప్రదేశ్‌లో సాధారణ నైరుతి బుతుపవనాల వర్షపాతం ఎంత?"

# Embed the question using the same E5 model
query_embedding = model.encode([f"query: {query}"])

# Search in the FAISS index
top_k = 3
distances, indices = index.search(np.array(query_embedding).astype("float32"), top_k)

# Show top matching text chunks
print("🧠 Top matching chunks:\n")
for i, idx in enumerate(indices[0]):
    print(f"{i+1}. {chunks[idx][:300]}...\n")


🔹 Sample of the loaded text:
శ్రీ శోభకృత్‌ నామ సంవత్సరము
వాతావరణం - పంటల ఉత్పత్తి - ధరల విశ్లేషణ

వాతావరణం :

ఆంధ్రప్రదేశ్‌ ప్రధానంగా వ్యవసాయ ఆధారిత రాష్ట్రం. రాష్ట్రంలోని పంటల విస్తీర్ణం, పంటల సరళి మరియు పంటల దిగుబడి ముఖ్యంగా నైరుతి బుతుపవనాల ప్రవేశం, విస్తరణ, పంట కాలంలో వీటి ద్వారా లభించే వర్షపాతం తదితర అంశాలపై ఆధారపడి ఉంటుంది. రాష్ట్రంలో జూన్‌ నుండి సెప్టెంబరు వరకు నైరుతి బుతుపవనాల ప్రభావం వలన, అక్టోబరు నుండి డిసెంబరు వరకు ఈశాన్య బుతుపవనాల ప్రభావం వలన వర్షాలు కురుస్తాయి. రాష్ట్రంలోని అన్ని ప్రాంతాలలో నైబుతి బుతుపవనాల ప్ర
🔹 Total chunks created: 2333
✅ Index loaded with 2333 vectors


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

🧠 Top matching chunks:

1. ఆంధ్రప్రదేశ్‌లో నైరుతి బుతుపవనాల సమయంలో (జూన్‌-సెప్టెంబరు, 2022) 574.7 మి.మీ. సాధారణ వర్షపాతమునకు గాను, 588.2 మి.మీ. (1.5%) వర్షపాతం నమోదు అయినది. ప్రాంతాల వారీగా విశ్లేషించి చూసినట్లయితే, నైరుతి బుతుపవనాల కాలంలో, ఉత్తర మరియు దక్షిణ కోస్తాలో -9.7 మరియు 1.7 శాతం చొప్పున రాయలసీమలో 9.4 శాతం చొప్పున సాధ...

2. 2022 వ సంవత్సరం దీర్ధకాలిక వాతావరణ సూచనలు - పరిశీలన :

2022వ సంవత్సరంలో భారత వాతావరణ విభాగం, న్యూ ఢిల్లీ వారు, నైరుతి బుతుపవనాల సమయంలో (జూన్‌-సెఫ్టెంబర్‌) సాధారణ వర్షపాతం నమోదయ్యే అవకాశం ఉందని దీర్ధకాలిక ముందస్తు సూచనలు ఇవ్వడం జరిగినది. 2022 జూన్‌ నుండి సెప్టెంబర్‌ వరకు కురిసిన వర్పపాత వివరాలను గమని...

3. రాష్ట్రంలో గడచిన ఐదు సంవత్సరాలలో నైరుతి బుతుపవనాల సమయంలో నాలుగు సంవత్సరాలు (2018, 2019, 2021, 2022) సాధారణ వర్షపాతం, ఒక సంవత్సరం (2020) సాధారణం కంటే అధిక వర్షపాతం నమోదైనది. అదే విధంగా ఈశాన్య బుతుపవనాల సమయంలో ఒక సంవత్సరం (2018) సాధారణం కంటే తక్కువ వర్షపాతం, రెండు సంవత్సరాలు (2019 మరియు 2022) సాధారణ వ...



In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "bigscience/bloomz-1b1"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id).to("cuda" if torch.cuda.is_available() else "cpu")

question = "2022లో ఆంధ్రప్రదేశ్‌లో సాధారణ నైరుతి బుతుపవనాల వర్షపాతం ఎంత?"
context = "\n\n".join([chunks[idx] for idx in indices[0]])

prompt = f"""కింది సమాచారం ఆధారంగా తెలుగులో సమాధానం ఇవ్వండి:

సమాచారం:
{context}

ప్రశ్న:
{question}

సమాధానం:"""

inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(model.device)
print(f"🧮 Prompt tokens: {inputs['input_ids'].shape[1]}")

output = model.generate(
    **inputs,
    max_new_tokens=200,  # Increase answer size
    do_sample=True,
    top_p=0.9,
    temperature=0.7,
    repetition_penalty=1.2,
    no_repeat_ngram_size=3,
    eos_token_id=tokenizer.eos_token_id
)

answer = tokenizer.decode(output[0], skip_special_tokens=True)
print("📌 BLOOMZ Answer:\n")
print(answer)


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


🧮 Prompt tokens: 413
📌 BLOOMZ Answer:

కింది సమాచారం ఆధారంగా తెలుగులో సమాధానం ఇవ్వండి:

సమాచారం:
ఆంధ్రప్రదేశ్‌లో నైరుతి బుతుపవనాల సమయంలో (జూన్‌-సెప్టెంబరు, 2022) 574.7 మి.మీ. సాధారణ వర్షపాతమునకు గాను, 588.2 మి.మీ. (1.5%) వర్షపాతం నమోదు అయినది. ప్రాంతాల వారీగా విశ్లేషించి చూసినట్లయితే, నైరుతి బుతుపవనాల కాలంలో, ఉత్తర మరియు దక్షిణ కోస్తాలో -9.7 మరియు 1.7 శాతం చొప్పున రాయలసీమలో 9.4 శాతం చొప్పున సాధారణ వర్షపాతం నమోదయినది (పట్టిక 1). జిల్లాల వారీగా విశ్లేషించి చూసినట్లయితే నైరుతి బుతుపవనాల కాలంలో, శ్రీకాకుళం (3. 6%), విజయనగరం (19.3%), పార్వతిపురం మన్యం (10.4%), అల్లూరి సీతారామరాజు (10. 2%),

2022 వ సంవత్సరం దీర్ధకాలిక వాతావరణ సూచనలు - పరిశీలన :

2022వ సంవత్సరంలో భారత వాతావరణ విభాగం, న్యూ ఢిల్లీ వారు, నైరుతి బుతుపవనాల సమయంలో (జూన్‌-సెఫ్టెంబర్‌) సాధారణ వర్షపాతం నమోదయ్యే అవకాశం ఉందని దీర్ధకాలిక ముందస్తు సూచనలు ఇవ్వడం జరిగినది. 2022 జూన్‌ నుండి సెప్టెంబర్‌ వరకు కురిసిన వర్పపాత వివరాలను గమనించినటైైతే ఆంధ్రప్రదేశ్‌లో సాధారణ వర్షపాతం నమోదైనది.

రాష్ట్రంలో గడచిన ఐదు సంవత్సరాలలో నైరుతి బుతుపవనాల సమయం

In [8]:
# Load fine-tuned BLOOM model
tokenizer = AutoTokenizer.from_pretrained("/kaggle/input/finetuned/bloom560m-finetuned/kaggle/working/bloom560m-finetuned")
model = AutoModelForCausalLM.from_pretrained("/kaggle/input/finetuned/bloom560m-finetuned/kaggle/working/bloom560m-finetuned").to("cuda" if torch.cuda.is_available() else "cpu")

# Use the same prompt as before
inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(model.device)
output = model.generate(
    **inputs,
    max_new_tokens=200,  # Increase answer size
    do_sample=True,
    top_p=0.9,
    temperature=0.7,
    repetition_penalty=1.2,
    no_repeat_ngram_size=3,
    eos_token_id=tokenizer.eos_token_id
)

fine_tuned_answer = tokenizer.decode(output[0], skip_special_tokens=True)
print("📌 Fine-tuned BLOOM Answer:\n")
print(fine_tuned_answer)


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


📌 Fine-tuned BLOOM Answer:

కింది సమాచారం ఆధారంగా తెలుగులో సమాధానం ఇవ్వండి:

సమాచారం:
ఆంధ్రప్రదేశ్‌లో నైరుతి బుతుపవనాల సమయంలో (జూన్‌-సెప్టెంబరు, 2022) 574.7 మి.మీ. సాధారణ వర్షపాతమునకు గాను, 588.2 మి.మీ. (1.5%) వర్షపాతం నమోదు అయినది. ప్రాంతాల వారీగా విశ్లేషించి చూసినట్లయితే, నైరుతి బుతుపవనాల కాలంలో, ఉత్తర మరియు దక్షిణ కోస్తాలో -9.7 మరియు 1.7 శాతం చొప్పున రాయలసీమలో 9.4 శాతం చొప్పున సాధారణ వర్షపాతం నమోదయినది (పట్టిక 1). జిల్లాల వారీగా విశ్లేషించి చూసినట్లయితే నైరుతి బుతుపవనాల కాలంలో, శ్రీకాకుళం (3. 6%), విజయనగరం (19.3%), పార్వతిపురం మన్యం (10.4%), అల్లూరి సీతారామరాజు (10. 2%),

2022 వ సంవత్సరం దీర్ధకాలిక వాతావరణ సూచనలు - పరిశీలన :

2022వ సంవత్సరంలో భారత వాతావరణ విభాగం, న్యూ ఢిల్లీ వారు, నైరుతి బుతుపవనాల సమయంలో (జూన్‌-సెఫ్టెంబర్‌) సాధారణ వర్షపాతం నమోదయ్యే అవకాశం ఉందని దీర్ధకాలిక ముందస్తు సూచనలు ఇవ్వడం జరిగినది. 2022 జూన్‌ నుండి సెప్టెంబర్‌ వరకు కురిసిన వర్పపాత వివరాలను గమనించినటైైతే ఆంధ్రప్రదేశ్‌లో సాధారణ వర్షపాతం నమోదైనది.

రాష్ట్రంలో గడచిన ఐదు సంవత్సరాలలో నైరుతి బుతుపవనాల సమయంలో నాలుగు స

In [2]:
!pip install -U langchain langchain-community faiss-cpu transformers sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 18.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 57.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 59.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 111.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 340.6/340.6 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 481.4/481.4 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.3/423.3 kB 24.0 MB/s eta 0:00:00
  Attempting uninstall: async-timeout
    Found existing installation: async-timeout 5.0.1
    Uninstalling async-timeout-5.0.1:
      Successfully uninstalled async-timeout-5.0.1
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.29.0
    Uninstalling huggingface-hub-0.29.0:
      Successfully uninstalled huggingface-hub-0.29.0
  Attempting uninstall: langchain-core

In [4]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from langchain.llms import HuggingFacePipeline
from transformers import AutoTokenizer, pipeline, AutoModelForCausalLM

# ✅ 1. Load the FAISS index and preprocessed chunks
index = faiss.read_index("/kaggle/input/fiass-index/telugu_index.faiss")
print(f"✅ FAISS index loaded with {index.ntotal} vectors")

# Load the original chunks that correspond to the index
with open("/kaggle/input/mini-tel/mini_tel.txt", "r", encoding="utf-8") as f:
    full_text = f.read()

from langchain.text_splitter import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_text(full_text)

# ✅ 2. Load embedding model
embed_model = SentenceTransformer("intfloat/multilingual-e5-base")

# ✅ 3. Load the BLOOM model
model_ckpt = "bigscience/bloomz-560m"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model = AutoModelForCausalLM.from_pretrained(model_ckpt)

text_gen_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=200,
    do_sample=True,
    temperature=0.7,
    top_p=0.95
)

llm = HuggingFacePipeline(pipeline=text_gen_pipeline)

# ✅ 4. Define the query
query = "2022లో ఆంధ్రప్రదేశ్‌లో సాధారణ నైరుతి బుతుపవనాల వర్షపాతం ఎంత?"

# ✅ 5. Embed the query
query_embedding = embed_model.encode([f"query: {query}"])
distances, indices = index.search(np.array(query_embedding).astype("float32"), k=3)

# ✅ 6. Retrieve top chunks
print("🔍 Retrieved Chunks:")
for i, idx in enumerate(indices[0]):
    print(f"\nChunk #{i+1} (index {idx}):")
    print(chunks[idx][:500])  # limit print length for readability

retrieved_contexts = "\n".join([chunks[idx] for idx in indices[0]])

# ✅ 7. Form the final prompt
final_prompt = f"""
దిగువ కాంటెక్స్ట్ ఆధారంగా, అడిగిన ప్రశ్నకు స్పష్టమైన, సంబంధిత సమాధానాన్ని తెలుగులో ఇవ్వండి.

కాంటెక్స్ట్:
{retrieved_contexts}

ప్రశ్న:
{query}

సమాధానం:
"""


response = llm(final_prompt, max_new_tokens=150)
print("🧠 Answer:", response.strip())

✅ FAISS index loaded with 2333 vectors


Device set to use cuda:0


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

🔍 Retrieved Chunks:

Chunk #1 (index 7):
ఆంధ్రప్రదేశ్‌లో నైరుతి బుతుపవనాల సమయంలో (జూన్‌-సెప్టెంబరు, 2022) 574.7 మి.మీ. సాధారణ వర్షపాతమునకు గాను, 588.2 మి.మీ. (1.5%) వర్షపాతం నమోదు అయినది. ప్రాంతాల వారీగా విశ్లేషించి చూసినట్లయితే, నైరుతి బుతుపవనాల కాలంలో, ఉత్తర మరియు దక్షిణ కోస్తాలో -9.7 మరియు 1.7 శాతం చొప్పున రాయలసీమలో 9.4 శాతం చొప్పున సాధారణ వర్షపాతం నమోదయినది (పట్టిక 1). జిల్లాల వారీగా విశ్లేషించి చూసినట్లయితే నైరుతి బుతుపవనాల కాలంలో, శ్రీకాకుళం (3. 6%), విజయనగరం (19.3%), పార్వతిపురం మన్యం (10.4%), అల్లూరి సీతారామరాజు (10. 2%),

Chunk #2 (index 6):
2022 వ సంవత్సరం దీర్ధకాలిక వాతావరణ సూచనలు - పరిశీలన :

2022వ సంవత్సరంలో భారత వాతావరణ విభాగం, న్యూ ఢిల్లీ వారు, నైరుతి బుతుపవనాల సమయంలో (జూన్‌-సెఫ్టెంబర్‌) సాధారణ వర్షపాతం నమోదయ్యే అవకాశం ఉందని దీర్ధకాలిక ముందస్తు సూచనలు ఇవ్వడం జరిగినది. 2022 జూన్‌ నుండి సెప్టెంబర్‌ వరకు కురిసిన వర్పపాత వివరాలను గమనించినటైైతే ఆంధ్రప్రదేశ్‌లో సాధారణ వర్షపాతం నమోదైనది.

Chunk #3 (index 13):
రాష్ట్రంలో గడచిన ఐదు సంవత్సరాలలో నైరుతి బుతుపవనాల సమయంలో నాలుగు సంవత్

<ipython-input-4-97b2b979d7c6>:68: LangChainDeprecationWarning: The method `BaseLLM.__call__` was deprecated in langchain-core 0.1.7 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = llm(final_prompt, max_new_tokens=150)


🧠 Answer: దిగువ కాంటెక్స్ట్ ఆధారంగా, అడిగిన ప్రశ్నకు స్పష్టమైన, సంబంధిత సమాధానాన్ని తెలుగులో ఇవ్వండి.

కాంటెక్స్ట్:
ఆంధ్రప్రదేశ్‌లో నైరుతి బుతుపవనాల సమయంలో (జూన్‌-సెప్టెంబరు, 2022) 574.7 మి.మీ. సాధారణ వర్షపాతమునకు గాను, 588.2 మి.మీ. (1.5%) వర్షపాతం నమోదు అయినది. ప్రాంతాల వారీగా విశ్లేషించి చూసినట్లయితే, నైరుతి బుతుపవనాల కాలంలో, ఉత్తర మరియు దక్షిణ కోస్తాలో -9.7 మరియు 1.7 శాతం చొప్పున రాయలసీమలో 9.4 శాతం చొప్పున సాధారణ వర్షపాతం నమోదయినది (పట్టిక 1). జిల్లాల వారీగా విశ్లేషించి చూసినట్లయితే నైరుతి బుతుపవనాల కాలంలో, శ్రీకాకుళం (3. 6%), విజయనగరం (19.3%), పార్వతిపురం మన్యం (10.4%), అల్లూరి సీతారామరాజు (10. 2%),
2022 వ సంవత్సరం దీర్ధకాలిక వాతావరణ సూచనలు - పరిశీలన :

2022వ సంవత్సరంలో భారత వాతావరణ విభాగం, న్యూ ఢిల్లీ వారు, నైరుతి బుతుపవనాల సమయంలో (జూన్‌-సెఫ్టెంబర్‌) సాధారణ వర్షపాతం నమోదయ్యే అవకాశం ఉందని దీర్ధకాలిక ముందస్తు సూచనలు ఇవ్వడం జరిగినది. 2022 జూన్‌ నుండి సెప్టెంబర్‌ వరకు కురిసిన వర్పపాత వివరాలను గమనించినటైైతే ఆంధ్రప్రదేశ్‌లో సాధారణ వర్షపాతం నమోదైనది.
రాష్ట్రంలో గడచిన ఐదు సంవత్సరాలలో నైరు

In [1]:
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM

# Load BLOOMZ 560M model from Hugging Face
model_name = "bigscience/bloomz-560m"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

generator = pipeline("text-generation", model=model, tokenizer=tokenizer)

context = "ఆంధ్రప్రదేశ్‌లో నైరుతి బుతుపవనాల సమయంలో (జూన్‌-సెప్టెంబరు, 2022) 574.7 మి.మీ. సాధారణ వర్షపాతమునకు గాను, 588.2 మి.మీ. (1.5%) వర్షపాతం నమోదు అయినది. ప్రాంతాల వారీగా విశ్లేషించి చూసినట్లయితే, నైరుతి బుతుపవనాల కాలంలో, ఉత్తర మరియు దక్షిణ కోస్తాలో -9.7 మరియు 1.7 శాతం చొప్పున రాయలసీమలో 9.4 శాతం చొప్పున సాధారణ వర్షపాతం నమోదయినది (పట్టిక 1). జిల్లాల వారీగా విశ్లేషించి చూసినట్లయితే నైరుతి బుతుపవనాల కాలంలో, శ్రీకాకుళం (3. 6%), విజయనగరం (19.3%), పార్వతిపురం మన్యం (10.4%), అల్లూరి సీతారామరాజు (10. 2%),"

prompt = f"""కాంటెక్స్ట్: {context}
ప్రశ్న: 2022 నైరుతి బుతుపవనాల వర్షపాతం లో ఎంత శాతం అదనంగా నమోదైంది?
సమాధానం:"""

output = generator(prompt, max_new_tokens=50, do_sample=False)

print(output[0]["generated_text"])


tokenizer_config.json:   0%|          | 0.00/222 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/715 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Device set to use cuda:0


కాంటెక్స్ట్: ఆంధ్రప్రదేశ్‌లో నైరుతి బుతుపవనాల సమయంలో (జూన్‌-సెప్టెంబరు, 2022) 574.7 మి.మీ. సాధారణ వర్షపాతమునకు గాను, 588.2 మి.మీ. (1.5%) వర్షపాతం నమోదు అయినది. ప్రాంతాల వారీగా విశ్లేషించి చూసినట్లయితే, నైరుతి బుతుపవనాల కాలంలో, ఉత్తర మరియు దక్షిణ కోస్తాలో -9.7 మరియు 1.7 శాతం చొప్పున రాయలసీమలో 9.4 శాతం చొప్పున సాధారణ వర్షపాతం నమోదయినది (పట్టిక 1). జిల్లాల వారీగా విశ్లేషించి చూసినట్లయితే నైరుతి బుతుపవనాల కాలంలో, శ్రీకాకుళం (3. 6%), విజయనగరం (19.3%), పార్వతిపురం మన్యం (10.4%), అల్లూరి సీతారామరాజు (10. 2%),
ప్రశ్న: 2022 నైరుతి బుతుపవనాల వర్షపాతం లో ఎంత శాతం అదనంగా నమోదైంది?
సమాధానం: 1.5%


In [1]:
# Step 1: Install dependencies
!pip install faiss-cpu
!pip install sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 48.8 MB/s eta 0:00:00:00:0100:01


In [2]:
!pip install git+https://github.com/anoopkunchukuttan/indic_nlp_library.git

  Cloning https://github.com/anoopkunchukuttan/indic_nlp_library.git to /tmp/pip-req-build-nhqonru0
  Running command git clone --filter=blob:none --quiet https://github.com/anoopkunchukuttan/indic_nlp_library.git /tmp/pip-req-build-nhqonru0
  Resolved https://github.com/anoopkunchukuttan/indic_nlp_library.git to commit 4cead0ae6c78fe9a19a51ef679f586206df9c476
  Preparing metadata (setup.py) ... done
  Created wheel for indic_nlp_library: filename=indic_nlp_library-0.92-py3-none-any.whl size=40777 sha256=0cd2004a44f3797198cb5b3afbe69c25d077cabf2db1cf73b55c1e50d7ab77f6
  Stored in directory: /tmp/pip-ephem-wheel-cache-qjzs9dy1/wheels/98/bc/25/381dc5529b731f558b894c7544c4f3ac12ab58b97de9c3b23d
Successfully built indic_nlp_library


In [4]:
!wget https://anoopkunchukuttan.github.io/indic_nlp_resources/indic_nlp_resources.zip
!unzip -q indic_nlp_resources.zip -d /kaggle/working/


--2025-04-12 13:56:18--  https://anoopkunchukuttan.github.io/indic_nlp_resources/indic_nlp_resources.zip
Resolving anoopkunchukuttan.github.io (anoopkunchukuttan.github.io)... 185.199.110.153, 185.199.111.153, 185.199.108.153, ...
Connecting to anoopkunchukuttan.github.io (anoopkunchukuttan.github.io)|185.199.110.153|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2025-04-12 13:56:18 ERROR 404: Not Found.

unzip:  cannot find or open indic_nlp_resources.zip, indic_nlp_resources.zip.zip or indic_nlp_resources.zip.ZIP.


In [3]:
import os
import pickle
import faiss
import numpy as np
from indicnlp import common
from indicnlp.tokenize import sentence_tokenize
from sentence_transformers import SentenceTransformer

# === Step 1: Set up Indic NLP Resources ===
INDIC_RESOURCES_PATH = "/kaggle/working/indic_nlp_resources"
common.set_resources_path(INDIC_RESOURCES_PATH)

# === Step 2: Load Telugu text ===
with open("/kaggle/input/mini-telu/mini_tel.txt", "r", encoding="utf-8") as f:
    full_text = f.read()

# === Step 3: Sentence Tokenization using Indic NLP ===
lang = "te"
chunks = sentence_tokenize.sentence_split(full_text, lang)
print(f"✅ Total Telugu sentence chunks: {len(chunks)}")
print("🔍 Sample sentence:", chunks[14])

# === Step 4: Save Chunks ===
output_dir = "/kaggle/working/faiss_store"
os.makedirs(output_dir, exist_ok=True)

chunks_path = os.path.join(output_dir, "chunks.pkl")
with open(chunks_path, "wb") as f:
    pickle.dump(chunks, f)
print(f"✅ Chunks saved at: {chunks_path}")

# === Step 5: Load Embedding Model ===
embed_model = SentenceTransformer("intfloat/multilingual-e5-base")

# === Step 6: Encode Chunks ===
embeddings = embed_model.encode([f"passage: {chunk}" for chunk in chunks], show_progress_bar=True)

# === Step 7: Build FAISS Index ===
dimension = embeddings.shape[1]
faiss_index = faiss.IndexFlatL2(dimension)
faiss_index.add(np.array(embeddings).astype("float32"))

# === Step 8: Save FAISS Index ===
index_path = os.path.join(output_dir, "telugu_index.faiss")
faiss.write_index(faiss_index, index_path)
print(f"✅ FAISS index saved at: {index_path} with {faiss_index.ntotal} vectors")


✅ Total Telugu sentence chunks: 8175
🔍 Sample sentence: జిల్లాల వారీగా విశ్లేషించి చూసినట్లయితే నైరుతి బుతుపవనాల కాలంలో, శ్రీకాకుళం (3. 6%), విజయనగరం (19.3%), పార్వతిపురం మన్యం (10.4%), అల్లూరి సీతారామరాజు (10. 2%), విశాఖపట్నం (-3. 1%) మరియు అనకాపల్లి (0. 1%) జిల్లాలలో సాధారణ వర్షపాతం నమోదు అయినది.
✅ Chunks saved at: /kaggle/working/faiss_store/chunks.pkl


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Batches:   0%|          | 0/256 [00:00<?, ?it/s]

✅ FAISS index saved at: /kaggle/working/faiss_store/telugu_index.faiss with 8175 vectors


In [4]:
import pickle
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM

# === Step 1: Load FAISS Index and Chunks ===
input_dir = "/kaggle/input/fiass-store"  # <-- Replace with your folder name
with open(f"{input_dir}/chunks.pkl", "rb") as f:
    chunks = pickle.load(f)

index = faiss.read_index(f"{input_dir}/telugu_index.faiss")
print(f"✅ Loaded index with {index.ntotal} vectors and {len(chunks)} chunks")

# === Step 2: Load Embedding Model ===
embed_model = SentenceTransformer("intfloat/multilingual-e5-base")

# === Step 3: Load BLOOMZ Model ===
model_name = "bigscience/bloomz-1b1"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
generator = pipeline("text-generation", model=model, tokenizer=tokenizer)

# === Step 4: Define a Query Function ===
def answer_query(query, k=1, max_tokens=50):
    # Embed the query
    query_vector = embed_model.encode([f"query: {query}"])
    
    # Search FAISS index
    D, I = index.search(np.array(query_vector).astype("float32"), k=k)
    
    # Show retrieved chunks (for debugging)
    print("📌 Retrieved Contexts:\n")
    for idx, i in enumerate(I[0]):
        print(f"[{idx+1}] {chunks[i]}\n{'-'*60}")
    
    # Combine retrieved chunks
    retrieved_context = "\n".join([chunks[i] for i in I[0]])
    
    # Build the prompt
    prompt = f"""కాంటెక్స్ట్: {retrieved_context}
ప్రశ్న: {query}
సమాధానం:"""
    
    # Generate the answer
    output = generator(prompt, max_new_tokens=max_tokens, do_sample=False)
    return output[0]["generated_text"]

# === Step 5: Test a Query ===
query = "అక్టోబర్ నుండి డిసెంబర్ వరకు వర్షాలు కురిసే వర్షకాలం పేరు ఏమిటి?"
response = answer_query(query)
print("✅ Final Answer:\n", response)


✅ Loaded index with 8175 vectors and 8175 chunks


tokenizer_config.json:   0%|          | 0.00/222 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/715 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.13G [00:00<?, ?B/s]

Device set to use cuda:0


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

📌 Retrieved Contexts:

[1] రాష్ట్రంలో జూన్‌ నుండి సెప్టెంబరు వరకు నైరుతి బుతుపవనాల ప్రభావం వలన, అక్టోబరు నుండి డిసెంబరు వరకు ఈశాన్య బుతుపవనాల ప్రభావం వలన వర్షాలు కురుస్తాయి.
------------------------------------------------------------
✅ Final Answer:
 కాంటెక్స్ట్: రాష్ట్రంలో జూన్‌ నుండి సెప్టెంబరు వరకు నైరుతి బుతుపవనాల ప్రభావం వలన, అక్టోబరు నుండి డిసెంబరు వరకు ఈశాన్య బుతుపవనాల ప్రభావం వలన వర్షాలు కురుస్తాయి.
ప్రశ్న: అక్టోబర్ నుండి డిసెంబర్ వరకు వర్షాలు కురిసే వర్షకాలం పేరు ఏమిటి?
సమాధానం: నైరుతి బుతుపవనాల ప్రభావం


In [5]:
query = "వరి పంటకి ఏ ఎరువులు వాడాలి?"
response = answer_query(query)
print("✅ Final Answer:\n", response)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

📌 Retrieved Contexts:

[1] పశువులు, గొర్రెలు, మేకలు మరియు పెరటి కోళ్ళ పెంపకము ద్వారా లభించే ఎరువును పంటపొలాలకువాడుకోవచ్చు.
------------------------------------------------------------
✅ Final Answer:
 కాంటెక్స్ట్: పశువులు, గొర్రెలు, మేకలు మరియు పెరటి కోళ్ళ పెంపకము ద్వారా లభించే ఎరువును పంటపొలాలకువాడుకోవచ్చు.
ప్రశ్న: వరి పంటకి ఏ ఎరువులు వాడాలి?
సమాధానం: ఎరువులు
